In [29]:
import nltk
import pandas as pd
import string

from nltk import PorterStemmer
from nltk.corpus import stopwords

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Descargá las dependencias necesarias de NLTK
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [24]:
categorias = ['sci.space', 'rec.autos']
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categorias,
    remove=('headers', 'footers', 'quotes'),
)

df = pd.DataFrame({'texto': newsgroups.data, 'etiqueta': newsgroups.target})
# Filtrá publicaciones vacías
df = df[df['texto'].str.strip() != ''].reset_index(drop=True)
print(f'Total de documentos cargados: {len(df)}')
print(df.head())

Total de documentos cargados: 1892
                                               texto  etiqueta
0  .. \n\nThese seem hardly like the groups to di...         1
1  It's been a while since I've reviewed a car bu...         0
2  \n\nnice theory.  too bad the MR2's never came...         0
3  \nIt was more than a theoretical concept; it w...         1
4  My wife and I looked at, and drove one last fa...         0


In [40]:
#1. Tokenizá y limpiá un texto de muestra: Tomá el primer documento del dataset, convertilo a minúsculas, eliminá los signos de puntuación y filtrá las palabras vacías (stopwords) en inglés usando nltk.

stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

first_document = df['texto'].iloc[0]

#Minusculas y tokenizacion
first_document.lower()
tokens = first_document.split()

tokens_limpios = []

tokens_limpios = [
    word.strip(string.punctuation)
    for word in tokens
    if word not in (stop_words or punctuation) and word.isalpha()
]

print(tokens_limpios)

['These', 'seem', 'hardly', 'like', 'groups', 'discuss', 'All', 'legitimate', 'power', 'enforce', 'rights', 'derives', 'consent', 'piece', 'Civilized', 'autonomous', 'computer', 'The', 'Constitution', 'made', 'people', 'trashed', 'us', 'sacred', 'scripture', 'rights', 'Our', 'come', 'And', 'I', 'sure', 'see', 'request', 'vote', 'trashing', 'Again', 'opinion', 'keep', 'rights', 'using', 'going']


In [52]:
#2. Aplicá Stemming y Lematización: Creá instancias de PorterStemmer y WordNetLemmatizer de nltk. Compará cómo transforman las palabras de tu lista de tokens limpios hacia su raíz.
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stem_text = [stemmer.stem(t) and t.lower() for t in tokens_limpios]
lemmatizer_text = [lemmatizer.lemmatize(t) and t.lower() for t in tokens_limpios]

stems = list(set(stem_text))
lemas = list(set(lemmatizer_text))

print('Stem text', stems)
print('Lemmatize text', lemas)

Stem text ['enforce', 'request', 'and', 'see', 'again', 'groups', 'legitimate', 'seem', 'discuss', 'rights', 'our', 'made', 'sacred', 'piece', 'trashing', 'these', 'people', 'i', 'autonomous', 'keep', 'trashed', 'the', 'computer', 'going', 'constitution', 'come', 'like', 'scripture', 'sure', 'consent', 'derives', 'opinion', 'all', 'hardly', 'us', 'power', 'vote', 'civilized', 'using']
Lemmatize text ['enforce', 'request', 'and', 'see', 'again', 'groups', 'legitimate', 'seem', 'discuss', 'rights', 'our', 'made', 'sacred', 'piece', 'trashing', 'these', 'people', 'i', 'autonomous', 'keep', 'trashed', 'the', 'computer', 'going', 'constitution', 'come', 'like', 'scripture', 'sure', 'consent', 'derives', 'opinion', 'all', 'hardly', 'us', 'power', 'vote', 'civilized', 'using']


In [58]:
# 3. Armá una representación Bag of Words (BoW): Utilizá CountVectorizer sobre los primeros 5 documentos para generar una matriz de frecuencias e imprimí el vocabulario construido.

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = df['texto'].iloc[:5]
vectorizer = CountVectorizer(vocabulary=lemas)

matriz = vectorizer.fit_transform(documents)

print(vectorizer.get_feature_names_out())
print(matriz.toarray())



['enforce' 'request' 'and' 'see' 'again' 'groups' 'legitimate' 'seem'
 'discuss' 'rights' 'our' 'made' 'sacred' 'piece' 'trashing' 'these'
 'people' 'i' 'autonomous' 'keep' 'trashed' 'the' 'computer' 'going'
 'constitution' 'come' 'like' 'scripture' 'sure' 'consent' 'derives'
 'opinion' 'all' 'hardly' 'us' 'power' 'vote' 'civilized' 'using']
[[  1   1   2   1   1   1   1   1   1   4   3   1   1   1   1   2   1   0
    1   1   1   6   1   1   1   1   1   1   1   1   1   1   1   1   1   1
    1   1   1]
 [  0   0  26   1   1   0   0   1   0   0   0   2   0   0   0   2   0   0
    0   0   0 109   1   2   0   2   5   0   0   0   0   0   6   1   0   4
    0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   2   0   0   0   0   2   0   0   0   0   0   0   0   0   0
    0   0   2]
 [  0   0   2   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   3   0   0   0   0   1   0   1   0   0   0   0   0   1   0
    0   0   0]
 [  

In [55]:
#4. Extraé N-Gramos (Unigramas + Bigramas): Configurá CountVectorizer(ngram_range=(1, 2)) para incluir combinaciones de 2 palabras consecutivas y observá el vocabulario generado.

vectorizer_n_gramas = CountVectorizer(vocabulary=lemas, ngram_range=(1, 2))

matriz_n_gramas = vectorizer_n_gramas.fit_transform(documents)

print(vectorizer.get_feature_names_out())

['enforce' 'request' 'and' 'see' 'again' 'groups' 'legitimate' 'seem'
 'discuss' 'rights' 'our' 'made' 'sacred' 'piece' 'trashing' 'these'
 'people' 'i' 'autonomous' 'keep' 'trashed' 'the' 'computer' 'going'
 'constitution' 'come' 'like' 'scripture' 'sure' 'consent' 'derives'
 'opinion' 'all' 'hardly' 'us' 'power' 'vote' 'civilized' 'using']


In [64]:
#5 Calculá la matriz TF-IDF: Utilizá TfidfVectorizer para ponderar la importancia de las palabras según la frecuencia de término e inversa en documentos ( TF×IDF ).

tf_idf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
matrix_tfidf = tf_idf_vectorizer.fit_transform(documents)

print(matrix_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 705 stored elements and shape (5, 669)>
  Coords	Values
  (0, 261)	0.11169367703171901
  (0, 320)	0.07799548889099528
  (0, 253)	0.13844143372566872
  (0, 161)	0.13844143372566872
  (0, 274)	0.13844143372566872
  (0, 315)	0.13844143372566872
  (0, 421)	0.11169367703171901
  (0, 182)	0.13844143372566872
  (0, 474)	0.5537657349026749
  (0, 152)	0.13844143372566872
  (0, 132)	0.13844143372566872
  (0, 251)	0.13844143372566872
  (0, 546)	0.13844143372566872
  (0, 413)	0.13844143372566872
  (0, 400)	0.13844143372566872
  (0, 111)	0.13844143372566872
  (0, 250)	0.13844143372566872
  (0, 351)	0.13844143372566872
  (0, 63)	0.13844143372566872
  (0, 128)	0.11169367703171901
  (0, 434)	0.13844143372566872
  (0, 294)	0.13844143372566872
  (0, 134)	0.13844143372566872
  (0, 408)	0.13844143372566872
  (0, 600)	0.13844143372566872
  :	:
  (4, 276)	0.13599401904856168
  (4, 360)	0.13599401904856168
  (4, 119)	0.13599401904856168
  (4, 51)	

In [65]:
#6. Calculá la Similitud Coseno entre documentos: Usá cosine_similarity de sklearn.metrics.pairwise para comparar la representación vectorial TFxIDF del primer documento con todos los demás documentos del dataset.

from sklearn.metrics.pairwise import cosine_similarity

similitudes = cosine_similarity(matrix_tfidf[0], matrix_tfidf).flatten()

print(similitudes)


[1.         0.02557636 0.05575009 0.03340951 0.01225493]


In [70]:
from sklearn.model_selection import train_test_split

#7. Separá los datos en conjunto de entrenamiento y evaluación: Usá train_test_split dividiendo el dataset en 80% entrenamiento y 20% prueba.

X_train, X_test, y_train, y_test = train_test_split(df['texto'], df['etiqueta'], test_size=0.2, random_state=42)

In [75]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

#8. Construí un Pipeline e integrá Vectorizador + Clasificador: Creá un Pipeline en Scikit-Learn que combine TfidfVectorizer (para convertir texto a vectores) y LogisticRegression como modelo clasificador.

pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression()),
])

In [76]:
#9. Entrená el Pipeline y realizá predicciones: Aplicá .fit() sobre el conjunto de entrenamiento y realizá las predicciones sobre el set de prueba usando .predict.

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [77]:
from sklearn.metrics import classification_report

#10. Evaluá las métricas de rendimiento: Imprimí el classification_report para analizar las métricas de Precision, Recall, F1-Score y Accuracy en el problema de clasificación de textos.

nombres_clases = newsgroups.target_names
print(classification_report(y_test, y_pred, target_names=nombres_clases))

              precision    recall  f1-score   support

   rec.autos       0.96      0.94      0.95       197
   sci.space       0.94      0.96      0.95       182

    accuracy                           0.95       379
   macro avg       0.95      0.95      0.95       379
weighted avg       0.95      0.95      0.95       379

